<a href="https://colab.research.google.com/github/shahaanabbas-13/Quantum-Diabetes-Prediction/blob/main/notebooks/06_VQC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Variational Quantum Classifier (VQC)
A compact four-qubit variational circuit trained on a stratified subset using PennyLane and a classical optimizer. Begin with a small subset and a small number of epochs; increase only if runtime permits.

In [1]:
%pip -q install pennylane scikit-learn

In [3]:
from pathlib import Path
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif, SelectKBest, RFE
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve)
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
# If running from a notebook in /notebooks, move to project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "dataset" / "diabetes.csv"
if not DATA_PATH.exists():
    # Colab fallback: upload diabetes.csv into the current session or mount Drive.
    DATA_PATH = Path("/content/diabetes.csv")
df = pd.read_csv(DATA_PATH)
TARGET = "Outcome"
FEATURES = [c for c in df.columns if c != TARGET]
INVALID_ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.feature_selection import SelectKBest,mutual_info_classif
from sklearn.metrics import classification_report,roc_auc_score
import torch
torch.set_num_threads(1)
X=df.drop(columns=TARGET).replace({c:{0:np.nan} for c in INVALID_ZERO_COLS})
y=df[TARGET].astype(int)
Xtr_raw,Xte_raw,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
imp=KNNImputer(n_neighbors=5,weights="distance")
Xtr=imp.fit_transform(Xtr_raw); Xte=imp.transform(Xte_raw)
sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)
sel=SelectKBest(mutual_info_classif,k=4)
Xtr=sel.fit_transform(Xtr,y_train); Xte=sel.transform(Xte)
mm=MinMaxScaler(feature_range=(-np.pi,np.pi))
Xtr=mm.fit_transform(Xtr); Xte=mm.transform(Xte)
# Keep runtime manageable: stratified subset
if len(Xtr) > 250:
    _, Xv, _, yv = train_test_split(
        Xtr, np.asarray(y_train), test_size=250, stratify=np.asarray(y_train),
        random_state=42
    )
else:
    Xv, yv = Xtr, np.asarray(y_train)
yv=np.asarray(yv,dtype=np.float64)
n_qubits=4; n_layers=2
dev=qml.device("default.qubit",wires=n_qubits)
@qml.qnode(dev,interface="torch",diff_method="backprop")
def circuit(x,weights):
    qml.AngleEmbedding(x,wires=range(n_qubits),rotation="Y")
    qml.StronglyEntanglingLayers(weights,wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))
weight_shape=qml.StronglyEntanglingLayers.shape(n_layers=n_layers,n_wires=n_qubits)
weights=torch.nn.Parameter(0.01*torch.randn(weight_shape,dtype=torch.float64))
opt=torch.optim.Adam([weights],lr=0.03)
X_tensor=torch.tensor(Xv,dtype=torch.float64)
y_tensor=torch.tensor(yv,dtype=torch.float64)
def loss_fn():
    outputs=torch.stack([circuit(x,weights) for x in X_tensor])
    # Map expectation [-1,1] to probability [0,1]
    probs=(1-outputs)/2
    return torch.mean((probs-y_tensor)**2)
for epoch in range(30):
    opt.zero_grad(); loss=loss_fn(); loss.backward(); opt.step()
    if (epoch+1)%5==0: print(f"Epoch {epoch+1:02d}, loss={loss.item():.4f}")
def predict_vqc(Xnew):
    with torch.no_grad():
        out=torch.stack([circuit(torch.tensor(x,dtype=torch.float64),weights) for x in Xnew])
        return ((1-out)/2).cpu().numpy()
vqc_scores=predict_vqc(Xte)
vqc_pred=(vqc_scores>=0.5).astype(int)
print(classification_report(y_test,vqc_pred,digits=3))
print("ROC-AUC:",roc_auc_score(y_test,vqc_scores))

Epoch 05, loss=0.2407
Epoch 10, loss=0.2290
Epoch 15, loss=0.2181
Epoch 20, loss=0.2098
Epoch 25, loss=0.2031
Epoch 30, loss=0.1977
              precision    recall  f1-score   support

           0      0.688     0.770     0.726       100
           1      0.452     0.352     0.396        54

    accuracy                          0.623       154
   macro avg      0.570     0.561     0.561       154
weighted avg      0.605     0.623     0.610       154

ROC-AUC: 0.6487037037037038


This is a prototype VQC and not a claim of quantum advantage. Save the trained circuit weights and preprocessing objects for reproducibility before ending the runtime.